# Rankings

Calibrated F1 score on the local test, for experiment 1 and experiment 2. Click a column header to sort. Capture each table into `results/leaderboard/`.


In [2]:
import csv
import re
from pathlib import Path

from IPython.display import HTML, display


def find_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "coco_mlc").is_dir():
            return candidate
    raise FileNotFoundError("Project root not found (src/coco_mlc). Open this notebook from the repository.")


REPO = find_repo()


def published_figures(models_py: Path):
    """Params, GFLOPs and ImageNet top-1 from the registry, without importing torch."""
    text = models_py.read_text(encoding="utf-8")
    pattern = re.compile(
        r'"([a-z0-9_]+)":\s*ModelInfo\(\s*'
        r'"[^"]+",\s*"[^"]+",\s*"[^"]+",\s*'
        r'([0-9.]+),\s*([0-9.]+),\s*([0-9.]+)'
    )
    return {
        key: {"params": float(params), "gflops": float(gflops), "top1": float(top1)}
        for key, params, gflops, top1 in pattern.findall(text)
    }


REGISTRY = published_figures(REPO / "src" / "coco_mlc" / "models.py")

DISPLAY = {
    "shufflenet_v2_x1_0": "ShuffleNet V2 ×1.0",
    "mobilenet_v3_large": "MobileNetV3-Large",
    "efficientnet_b0": "EfficientNet-B0",
    "resnet18": "ResNet18",
    "resnet50": "ResNet50",
    "efficientnet_b4": "EfficientNet-B4",
    "convnext_tiny": "ConvNeXt-Tiny",
    "swin_t": "Swin-T",
    "maxvit_t": "MaxViT-T",
    "efficientnet_v2_s": "EfficientNetV2-S",
    "vgg16": "VGG16",
}


def notebook_batch_size(path: Path):
    match = re.search(r"BATCH_SIZE\s*=\s*(\d+)", path.read_text(encoding="utf-8"))
    return int(match.group(1)) if match else None


def calibrated_local_f1(path: Path):
    """Calibrated F1 score on the local-test row. French and English headers."""
    with path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.reader(handle))
    if len(rows) < 2:
        return None
    header = rows[0]
    f1_idx = None
    for i, name in enumerate(header):
        folded = name.casefold().replace("é", "e")
        if "f1" in folded and "calibr" in folded:
            f1_idx = i
            break
    if f1_idx is None:
        return None
    for row in rows[1:]:
        label = row[0].casefold()
        if "local" in label:
            try:
                return float(row[f1_idx])
            except ValueError:
                return None
    return None


def ranking_rows(notebook_dir: Path, output_dir: Path):
    rows = []
    for notebook in sorted(notebook_dir.glob("exp_*.ipynb")):
        key = notebook.stem[4:] if notebook.stem.startswith("exp_") else notebook.stem
        info = REGISTRY[key]
        generalization = output_dir / key / "generalization.csv"
        f1 = calibrated_local_f1(generalization) if generalization.is_file() else None
        rows.append(
            {
                "name": DISPLAY.get(key, key),
                "f1": f1,
                "params": info["params"],
                "gflops": info["gflops"],
                "top1": info["top1"],
                "batch": notebook_batch_size(notebook),
            }
        )
    rows.sort(key=lambda row: (row["f1"] is None, -(row["f1"] or 0.0)))
    return rows


def show_ranking(rows, table_id: str):
    def cell(value, text, kind):
        if value is None:
            return f'<td data-kind="{kind}" data-empty="1"></td>'
        return f'<td data-kind="{kind}" data-value="{value}">{text}</td>'

    body = []
    for row in rows:
        f1_text = "" if row["f1"] is None else f"{row['f1']:.3f}"
        body.append(
            "<tr>"
            f'<td data-kind="text" data-value="{row["name"]}">{row["name"]}</td>'
            + cell(row["f1"], f1_text, "num")
            + cell(row["params"], f"{row['params']:.1f}", "num")
            + cell(row["gflops"], f"{row['gflops']:.2f}", "num")
            + cell(row["top1"], f"{row['top1']:.2f}", "num")
            + cell(row["batch"], str(row["batch"]), "num")
            + "</tr>"
        )

    html = f"""
<style>
#{table_id} {{ border-collapse: collapse; font-size: 14px; }}
#{table_id} th, #{table_id} td {{ border-bottom: 1px solid #ddd; padding: 6px 12px; }}
#{table_id} th {{ cursor: pointer; text-align: left; white-space: nowrap; }}
#{table_id} td:not(:first-child), #{table_id} th:not(:first-child) {{ text-align: right; }}
#{table_id} th.sorted {{ text-decoration: underline; }}
</style>
<table id="{table_id}">
<thead><tr>
<th data-kind="text">Architecture</th>
<th data-kind="num" class="sorted" data-dir="desc">F1 score</th>
<th data-kind="num">Params (M)</th>
<th data-kind="num">GFLOPs</th>
<th data-kind="num">ImageNet top-1 (%)</th>
<th data-kind="num">Batch size in our runs</th>
</tr></thead>
<tbody>
{''.join(body)}
</tbody></table>
<script>
(function () {{
  const table = document.getElementById("{table_id}");
  const headers = Array.from(table.tHead.rows[0].cells);
  function valueOf(cell) {{
    if (cell.dataset.empty === "1") return null;
    if (cell.dataset.kind === "num") return parseFloat(cell.dataset.value);
    return cell.dataset.value.toLowerCase();
  }}
  function sortBy(index, direction) {{
    const body = table.tBodies[0];
    const rows = Array.from(body.rows);
    rows.sort(function (a, b) {{
      const va = valueOf(a.cells[index]);
      const vb = valueOf(b.cells[index]);
      if (va === null && vb === null) return 0;
      if (va === null) return 1;
      if (vb === null) return -1;
      if (va < vb) return direction === "asc" ? -1 : 1;
      if (va > vb) return direction === "asc" ? 1 : -1;
      return 0;
    }});
    rows.forEach(function (row) {{ body.appendChild(row); }});
    headers.forEach(function (th, i) {{
      th.classList.toggle("sorted", i === index);
      th.dataset.dir = i === index ? direction : "";
    }});
  }}
  headers.forEach(function (th, index) {{
    th.addEventListener("click", function () {{
      const next = th.dataset.dir === "desc" ? "asc" : "desc";
      sortBy(index, next);
    }});
  }});
}})();
</script>
"""
    display(HTML(html))


## Experiment 1


In [3]:
rows = ranking_rows(
    REPO / "notebooks" / "experiments" / "1",
    REPO / "outputs" / "notebooks",
)
show_ranking(rows, "exp1-ranking")
print("Capture this table as results/leaderboard/experiment_1_ranking.png")


Architecture,F1 score,Params (M),GFLOPs,ImageNet top-1 (%),Batch size in our runs
MaxViT-T,0.714,30.9,5.56,83.70,8
ConvNeXt-Tiny,0.675,28.6,4.46,82.52,64
EfficientNetV2-S,0.671,21.5,8.37,84.23,16
Swin-T,0.670,28.3,4.49,81.47,64
ResNet50,0.664,25.6,4.09,80.86,96
EfficientNet-B0,0.619,5.3,0.39,77.69,64
MobileNetV3-Large,0.613,5.5,0.22,75.27,96
ResNet18,0.595,11.7,1.81,69.76,128
EfficientNet-B4,0.576,19.3,4.39,83.38,16
ShuffleNet V2 ×1.0,0.482,2.3,0.14,69.36,128


Capture this table as results/leaderboard/experiment_1_ranking.png


## Experiment 2


In [4]:
rows = ranking_rows(
    REPO / "notebooks" / "experiments" / "2",
    REPO / "outputs" / "notebooks" / "2",
)
show_ranking(rows, "exp2-ranking")
print("Capture this table as results/leaderboard/experiment_2_ranking.png")


Architecture,F1 score,Params (M),GFLOPs,ImageNet top-1 (%),Batch size in our runs
MaxViT-T,0.694,30.9,5.56,83.70,8
ConvNeXt-Tiny,0.681,28.6,4.46,82.52,64


Capture this table as results/leaderboard/experiment_2_ranking.png
